In [1]:
import mlflow
import polars as pl
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
from torch.utils.data import DataLoader, Dataset
from torchinfo import summary
from tqdm.notebook import tqdm
from transformers import AutoModel, AutoTokenizer
from transformers.models.bert.modeling_bert import BertModel
from mlflow.models import infer_signature

# Types
from transformers.models.bert.tokenization_bert_fast import BertTokenizerFast

import matplotlib.pyplot as plt
from src.config import MPL_STYLE_DIR, PROCESSED_DATA_DIR
from src.db import PBWarehouse
from src.models import Tweet

mlflow.set_tracking_uri("http://192.168.100.203:5000")
mlflow.set_experiment("[CAPSTONE-2] hatebert-finetuning")

warehouse = PBWarehouse()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

2025-07-12 09:18:35.058 | INFO     | src.config:<module>:26 - Loaded environment variables from /home/iragca/Documents/github/capstone-project-2/.env
2025-07-12 09:18:35.058 | INFO     | src.config:<module>:60 - PROJECT_ROOT: /home/iragca/Documents/github/capstone-project-2
2025-07-12 09:18:35.059 | INFO     | src.config:<module>:61 - DATA_DIR: /home/iragca/Documents/github/capstone-project-2/data
2025/07/12 09:18:35 INFO mlflow.tracking.fluent: Experiment with name '[CAPSTONE-2] hatebert-finetuning' does not exist. Creating a new experiment.


device(type='cuda')

In [2]:
data = warehouse.client.collection("tweets_v2").get_full_list(
    query_params={
        "filter": (
            "has_blm_hashtag = true || "
            "is_reply_to_blm = true && "
            "creation_date >= '2020-03-26' "
            "&& creation_date <= '2020-07-24' "
            "&& language = 'en'",
        )
    }
)
tweets: list[Tweet] = [Tweet(**r.__dict__) for r in data]

In [3]:
df = pl.DataFrame(
    [t.model_dump() for t in tweets],
    schema={
        "tweet_id": pl.Utf8,
        "text": pl.Utf8,
        "status_link": pl.Utf8,
        "user_id": pl.Utf8,
        "is_extremist": pl.Boolean,
        "is_annotated": pl.Boolean,
        "in_reply_to_status_link": pl.Utf8,
        "in_reply_to_status_id": pl.Utf8,
        "bookmark_count": pl.Int64,
        "views": pl.Int64,
        "retweet_count": pl.Int64,
        "favorite_count": pl.Int64,
        "reply_count": pl.Int64,
        "quote_count": pl.Int64,
        "conversation_id": pl.Utf8,
        "retweet_tweet_id": pl.Utf8,
        "quoted_status_id": pl.Utf8,
        "community_note": pl.Utf8,
        "language": pl.Utf8,
        "source": pl.Utf8,
        "creation_date": pl.Utf8,
        "has_blm_hashtag": pl.Boolean,
        "fetched_replies": pl.Boolean,
        "is_reply_to_blm": pl.Boolean,
    },
).with_columns(
    pl.col("creation_date").str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S%.3fZ")
)
df

tweet_id,text,status_link,user_id,is_extremist,is_annotated,in_reply_to_status_link,in_reply_to_status_id,bookmark_count,views,retweet_count,favorite_count,reply_count,quote_count,conversation_id,retweet_tweet_id,quoted_status_id,community_note,language,source,creation_date,has_blm_hashtag,fetched_replies,is_reply_to_blm
str,str,str,str,bool,bool,str,str,i64,i64,i64,i64,i64,i64,str,str,str,str,str,str,datetime[ms],bool,bool,bool
"""1269916756719685632""","""@TRCdocumentary @Blklivesmatte…","""https://x.com/hypersoses/statu…","""1264846057906806786""",false,false,"""https://x.com/TRCdocumentary/s…","""1266515065190170625""",0,0,0,0,0,0,"""1266484388491116550""",null,"""""","""""","""en""","""""",2020-06-08 08:58:43,false,false,true
"""1277976913743503365""","""@terrycrews @jonathanwsabin Th…","""https://x.com/TheJaredMonroe/s…","""23719684""",false,false,"""https://x.com/terrycrews/statu…","""1277955144332668930""",23,0,156,4860,334,28,"""1277955144332668930""",null,"""""","""""","""en""","""""",2020-06-30 14:46:54,true,true,true
"""1285714950896443395""","""Join Bernie and Sunrise leader…","""https://x.com/SunriseMvmtHTX/s…","""1081973336010309632""",false,false,"""""","""""",0,0,0,2,0,0,"""1285714950896443395""",null,"""""","""""","""en""","""Hootsuite Inc.""",2020-07-21 23:15:06,true,false,false
"""1284209535561818112""","""@prageru It's not the women's …","""https://x.com/chekmate111/stat…","""190751291""",false,false,"""https://x.com/prageru/status/1…","""1284205227319730176""",0,0,0,4,1,0,"""1284205227319730176""",null,"""""","""""","""en""","""""",2020-07-17 19:33:07,false,false,true
"""1575540451486011392""","""Imagine being homeless, then t…","""https://x.com/UnionBaddieMatt/…","""1535727235155607553""",false,false,"""""","""""",0,0,1,5,1,0,"""1575540451486011392""",null,"""""","""""","""en""","""Twitter for Android""",2022-09-29 17:38:10,true,true,false
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""1243348301900140544""","""You shouldn’t get down to the …","""https://x.com/great_african/st…","""1082588920414720000""",false,false,"""""","""""",0,0,0,1,0,0,"""1243348301900140544""",null,"""""","""""","""en""","""Twitter for iPhone""",2020-03-27 01:25:10,true,false,false
"""1269549033053290499""","""@AmberLotus21 @David76583783 D…","""https://x.com/ActivelyWoke1/st…","""553976157""",false,false,"""https://x.com/AmberLotus21/sta…","""1266926269876195328""",0,0,1,1,1,0,"""1266926269876195328""",null,"""""","""""","""en""","""""",2020-06-07 08:37:31,false,true,true
"""1242028539546828801""","""Overseas #American #Voters Req…","""https://x.com/DemsAbroadDE/sta…","""860520162939850752""",false,false,"""""","""""",0,0,1,1,0,0,"""1242028539546828801""",null,"""""","""""","""en""","""SocialPilot.co""",2020-03-23 10:00:54,true,false,false


In [4]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "Hate-speech-CNERG/bert-base-uncased-hatexplain"  # HATEBERT on Hugging Face

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

In [9]:
inputs = tokenizer("I hate black people", return_tensors="pt")
outputs = model(**inputs)

logits = outputs.logits

probs = torch.nn.functional.softmax(logits, dim=-1)
predicted_class = torch.argmax(probs, dim=-1)

print(f"Logits: {logits}")
print(f"Probabilities: {probs}")
print(f"Predicted class: {predicted_class.item()}")

Logits: tensor([[-1.0975,  0.3231, -0.1099]], grad_fn=<AddmmBackward0>)
Probabilities: tensor([[0.1278, 0.5291, 0.3431]], grad_fn=<SoftmaxBackward0>)
Predicted class: 1


In [10]:
test_df = df.select(
    [
        pl.col("text"),
    ]
).with_columns(pl.lit(None).alias("is_hateful").cast(pl.Int8))
test_df

text,is_hateful
str,i8
"""@TRCdocumentary @Blklivesmatte…",null
"""@terrycrews @jonathanwsabin Th…",null
"""Join Bernie and Sunrise leader…",null
"""@prageru It's not the women's …",null
"""Imagine being homeless, then t…",null
…,…
"""You shouldn’t get down to the …",null
"""@AmberLotus21 @David76583783 D…",null
"""Overseas #American #Voters Req…",null


In [11]:
def classify_text(text: str) -> int:
    inputs = tokenizer(text, return_tensors="pt")
    outputs = model(**inputs)
    logits = outputs.logits
    probs = torch.nn.functional.softmax(logits, dim=-1)
    predicted_class = torch.argmax(probs, dim=-1)
    return predicted_class.item()


In [12]:
classify_text("I hate you")

1

In [13]:
test_df = df.with_columns(
    pl.col("text").map_elements(lambda x: classify_text(x), return_dtype=pl.Int8).alias("is_hateful")
)

In [14]:
test_df.write_csv(PROCESSED_DATA_DIR / "hatebert-test.csv")
# test_df = pl.read_csv(PROCESSED_DATA_DIR / "hatebert-test.csv")

In [15]:
test_df

tweet_id,text,status_link,user_id,is_extremist,is_annotated,in_reply_to_status_link,in_reply_to_status_id,bookmark_count,views,retweet_count,favorite_count,reply_count,quote_count,conversation_id,retweet_tweet_id,quoted_status_id,community_note,language,source,creation_date,has_blm_hashtag,fetched_replies,is_reply_to_blm,is_hateful
str,str,str,str,bool,bool,str,str,i64,i64,i64,i64,i64,i64,str,str,str,str,str,str,datetime[ms],bool,bool,bool,i8
"""1269916756719685632""","""@TRCdocumentary @Blklivesmatte…","""https://x.com/hypersoses/statu…","""1264846057906806786""",false,false,"""https://x.com/TRCdocumentary/s…","""1266515065190170625""",0,0,0,0,0,0,"""1266484388491116550""",null,"""""","""""","""en""","""""",2020-06-08 08:58:43,false,false,true,1
"""1277976913743503365""","""@terrycrews @jonathanwsabin Th…","""https://x.com/TheJaredMonroe/s…","""23719684""",false,false,"""https://x.com/terrycrews/statu…","""1277955144332668930""",23,0,156,4860,334,28,"""1277955144332668930""",null,"""""","""""","""en""","""""",2020-06-30 14:46:54,true,true,true,1
"""1285714950896443395""","""Join Bernie and Sunrise leader…","""https://x.com/SunriseMvmtHTX/s…","""1081973336010309632""",false,false,"""""","""""",0,0,0,2,0,0,"""1285714950896443395""",null,"""""","""""","""en""","""Hootsuite Inc.""",2020-07-21 23:15:06,true,false,false,1
"""1284209535561818112""","""@prageru It's not the women's …","""https://x.com/chekmate111/stat…","""190751291""",false,false,"""https://x.com/prageru/status/1…","""1284205227319730176""",0,0,0,4,1,0,"""1284205227319730176""",null,"""""","""""","""en""","""""",2020-07-17 19:33:07,false,false,true,1
"""1575540451486011392""","""Imagine being homeless, then t…","""https://x.com/UnionBaddieMatt/…","""1535727235155607553""",false,false,"""""","""""",0,0,1,5,1,0,"""1575540451486011392""",null,"""""","""""","""en""","""Twitter for Android""",2022-09-29 17:38:10,true,true,false,1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""1243348301900140544""","""You shouldn’t get down to the …","""https://x.com/great_african/st…","""1082588920414720000""",false,false,"""""","""""",0,0,0,1,0,0,"""1243348301900140544""",null,"""""","""""","""en""","""Twitter for iPhone""",2020-03-27 01:25:10,true,false,false,1
"""1269549033053290499""","""@AmberLotus21 @David76583783 D…","""https://x.com/ActivelyWoke1/st…","""553976157""",false,false,"""https://x.com/AmberLotus21/sta…","""1266926269876195328""",0,0,1,1,1,0,"""1266926269876195328""",null,"""""","""""","""en""","""""",2020-06-07 08:37:31,false,true,true,1
"""1242028539546828801""","""Overseas #American #Voters Req…","""https://x.com/DemsAbroadDE/sta…","""860520162939850752""",false,false,"""""","""""",0,0,1,1,0,0,"""1242028539546828801""",null,"""""","""""","""en""","""SocialPilot.co""",2020-03-23 10:00:54,true,false,false,1


In [16]:
test_df.filter(pl.col("is_hateful") == 2).select(
    [pl.col("text"), pl.col("is_hateful")],
).to_pandas()

,text,is_hateful
0,@ukblm This sounds like a fairytale anyway.\n\...,2
1,Nice! Black Lives Matter doesn't even bother t...,2
2,@Trace___65roses So a career criminal is the n...,2
3,@BLMSeattleKC Eat a dick,2
4,I have to say: @UncleTomDoc Part II is AMAZING...,2
...,...,...
867,White gays need to start making good on the #B...,2
868,@johncardillo @gatewaypundit #BlackLivesMatter...,2
869,@ScotsmanGrumpy The gaming community will neve...,2
870,@MurielBowser The racist trying to sue you in ...,2


In [17]:
test_df.filter(pl.col("is_hateful") == 1).select(
    [pl.col("text"), pl.col("is_hateful")],
).to_pandas()

,text,is_hateful
0,@TRCdocumentary @Blklivesmatter only shits,1
1,@terrycrews @jonathanwsabin The very fact that...,1
2,Join Bernie and Sunrise leaders tomorrow at 7:...,1
3,@prageru It's not the women's body. It is a se...,1
4,"Imagine being homeless, then the proletarian r...",1
...,...,...
41246,LOUnited March today with @OleMissFB and other...,1
41247,You shouldn’t get down to the mud just to prov...,1
41248,@AmberLotus21 @David76583783 Did the police of...,1
41249,Overseas #American #Voters Request your ballot...,1


In [18]:
test_df.filter(pl.col("is_hateful") == 0).select(
    [pl.col("text"), pl.col("is_hateful")],
).to_pandas()

,text,is_hateful
0,@lausanhk @b9AcE @rhaphiikii Imperialist tools...,0
1,I gained the Covid 15 and still working multip...,0
2,@7NewsBrisbane burning our flag. no good cunt'...,0
3,@CameronRidle DNR filed charges with Prosecuto...,0
4,@CopWithAttitude I have a bad feeling they wou...,0
...,...,...
73,Goldie Star chokes on Big Black Cock 🙋🏼‍♀️👄🍆♠️...,0
74,Four years on from the @hackneygazette's headl...,0
75,"If your defeat makes you horny, you are owned....",0
76,@realNick_777 @malibujonfanti @RudyGiuliani On...,0


In [6]:
def has_hateful_words(text: str) -> bool:

    terms = [
        "bnwo",
        "bbc",
        "blacksupremacy",
        "queenofspades",
        "blacked",
        "qos"
    ]

    return any(term in text.lower() for term in terms)

hate_df = test_df.with_columns(
    pl.col("text").map_elements(lambda x: has_hateful_words(x), return_dtype=pl.Boolean).alias("has_hateful_words")
)


In [ ]:
hate_df["has_hateful_words"]

has_hateful_words,count
bool,u32
true,590
false,41845


In [22]:
hate_df = test_df.with_columns(
    pl.col("creation_date").str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S%.3fZ")  
    )

InvalidOperationError: conversion from `str` to `datetime[ms]` failed in column 'creation_date' for 226 out of 226 values: ["2020-06-03T14:19:15.000", "2020-06-12T15:06:50.000", … "2020-06-19T19:08:21.000"]

You might want to try:
- setting `strict=False` to set values that cannot be converted to `null`
- using `str.strptime`, `str.to_date`, or `str.to_datetime` and providing a format string

In [21]:
hate_df

tweet_id,text,status_link,user_id,is_extremist,is_annotated,in_reply_to_status_link,in_reply_to_status_id,bookmark_count,views,retweet_count,favorite_count,reply_count,quote_count,conversation_id,retweet_tweet_id,quoted_status_id,community_note,language,source,creation_date,has_blm_hashtag,fetched_replies,is_reply_to_blm,is_hateful
i64,str,str,i64,bool,bool,str,str,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,str,datetime[ms],bool,bool,bool,i64
1286447415080312833,"""Heyhey if you are someone/know…","""https://x.com/ChuckMockler/sta…",347913343,false,false,"""""","""""",0,0,1,2,0,0,1286447415080312833,"""""","""""","""""","""en""","""Twitter for iPhone""",null,true,true,false,1
1286440938106163203,"""Thank you @VoteAdamMedrano for…","""https://x.com/AdamBazaldua/sta…",833871911272722433,false,false,"""""","""""",0,0,4,16,4,1,1286440938106163203,"""""","""""","""""","""en""","""Twitter for iPhone""",null,true,true,false,1
1286450496329285632,"""I have ten-ish years of experi…","""https://x.com/LilyShumarKray/s…",773036078,false,false,"""https://x.com/LilyShumarKray/s…","""1286450494349533184""",0,0,0,3,0,0,1286450494349533184,"""""","""""","""""","""en""","""""",null,false,true,true,1
1286449764507291650,"""Opening day is underway for @m…","""https://x.com/HarrisWarRoom/st…",1148212967332204544,false,false,"""""","""""",0,0,10,50,0,1,1286449764507291650,"""""","""""","""""","""en""","""Twitter Web App""",null,true,true,false,1
1286439076233650177,"""People are either being Dof an…","""https://x.com/SoliPhilander/st…",108571906,false,false,"""""","""""",0,0,0,7,0,0,1286439076233650177,"""""","""""","""""","""en""","""Twitter for Android""",null,true,true,false,1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
1275947325983244288,"""@johncardillo This is such BS.…","""https://x.com/MainStreetMuse/s…",179785566,false,false,"""https://x.com/johncardillo/sta…","""1275946651308392450""",0,0,0,2,0,0,1275946651308392450,"""""","""""","""""","""en""","""""",null,false,false,true,1
1269056990178938880,"""@rebeinstein ... the only good…","""https://x.com/alanekennedylaw/…",2496399067,false,false,"""https://x.com/rebeinstein/stat…","""1269055716972564481""",0,0,0,3,1,0,1269055716972564481,"""""","""""","""""","""en""","""""",null,false,false,true,1
1273000292720668672,"""@tomselliott @dbongino @thread…","""https://x.com/MSMCali/status/1…",728434941843804161,false,false,"""https://x.com/tomselliott/stat…","""1272993765670739969""",0,0,0,1,1,0,1272986923951407106,"""""","""""","""""","""en""","""""",null,false,false,true,1


In [15]:
sorted_df = hate_df.filter(pl.col("has_hateful_words")).sort(pl.col("creation_date")).group_by_dynamic("creation_date", every="1m").agg(
    [
        pl.col("tweet_id").count().alias("total_tweets"),
    ])

ComputeError: null values in dynamic group_by not supported, fill nulls.